# Multipart identity check

What actually uniquely identifies one multipart-message group?

- SS7 raw fields: `sarref` / `msg_part` / `msg_parts`, joined against `b_number` / `msisdn` (the raw sender/receiver ids - superseded by `calling_gt`/`called_gt` in the canonical mapping, but those aren't populated with the same raw values, so check against the true raw ids here).
- SMPP raw fields: `sar_ref` / `sar_msg_parts` / `sar_msg_part`, joined against `oa` / `da` (SMPP's equivalent of b_number/msisdn - originator/destination address).

**Finding, verified against real data (one SS7 file, `stg_ss7_20260802_0000.csv`):**
no combination of ID columns alone is enough, and adding more of them (e.g.
`imsi`) doesn't help:
- `sarref` alone: 254 distinct values across 3,899 multipart rows, one
  value shared by up to 239 unrelated rows - basically a small rotating
  counter, not a per-message id.
- `(b_number, msisdn, sarref)` together: 1,479 distinct combos - much
  better, but **324/1,479 (22%) still collide** - the exact same sender/
  receiver pair reuses the exact same sarref for two or three genuinely
  separate messages sent close together in time (real example: three
  `msg_part=1` rows, same b_number/msisdn/sarref/**imsi**, ~1 minute
  apart - imsi is identical across all three since it's the same
  physical device, so it can't split them apart either).
- The actual fix is a **time-bounded** grouping, not more identity
  columns: within one `(b_number, msisdn, sarref)` key, start a NEW group
  whenever `msg_part` resets to a value it's already seen, or the gap
  since the previous part exceeds a threshold (30s used below). That
  alone drops the collision rate from 22% to **0/2,376 groups**. This is
  exactly the gap `features/message_reassembly.py`'s own docstring
  already flags ("does not include a time window... add a max-time-gap
  bound here") - checked here directly against raw data instead of
  taking that caveat on faith.

In [4]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if not (project_root / "ingestion").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

# Narrowed to just the columns this check needs - loading all 30+ raw
# columns across every file would multiply memory for nothing here.
# time_stamp + imsi added: needed to test the time-bounded grouping and to
# confirm imsi doesn't disambiguate same-subscriber collisions (see markdown).
SS7_COLS = ["time_stamp", "sarref", "msg_part", "msg_parts", "b_number", "msisdn", "imsi", "calling_gt", "called_gt"]
SMPP_COLS = ["time_stamp", "sar_ref", "sar_msg_parts", "sar_msg_part", "oa", "da"]

ss7_files = sorted((project_root / "data" / "raw" / "SS7").rglob("*.csv"))
smpp_files = sorted((project_root / "data" / "raw" / "SMPP").rglob("*.csv"))
print(f"{len(ss7_files)} SS7 files, {len(smpp_files)} SMPP files found")

ss7_raw = pd.concat(
    [pd.read_csv(f, usecols=lambda c: c in SS7_COLS, low_memory=False) for f in ss7_files],
    ignore_index=True,
)
smpp_raw = pd.concat(
    [pd.read_csv(f, usecols=lambda c: c in SMPP_COLS, low_memory=False) for f in smpp_files],
    ignore_index=True,
)
print(f"SS7 raw rows: {len(ss7_raw)}")
print(f"SMPP raw rows: {len(smpp_raw)}")

48 SS7 files, 48 SMPP files found
SS7 raw rows: 14685436
SMPP raw rows: 14101168


In [ ]:
def time_bounded_groups(df, key_cols, part_col, time_col, max_gap):
    """
    Vectorized version of: walk rows sorted by (key, time); start a NEW
    group whenever the key changes, OR msg_part resets to a value <= the
    previous part seen for that key (a real message's parts always go
    strictly 1, 2,
      3... - a repeat/reset means a DIFFERENT message reused
    the same key), OR the time gap since the previous part exceeds
    max_gap. Returns df sorted by (key, time) with a `group_id` column.
    """
    d = df.sort_values(key_cols + [time_col]).reset_index(drop=True)
    key_changed = (d[key_cols] != d[key_cols].shift()).any(axis=1)
    same_key = ~key_changed
    part_reset = same_key & (d[part_col] <= d[part_col].shift())
    gap_too_big = same_key & ((d[time_col] - d[time_col].shift()) > max_gap)
    d["group_id"] = (key_changed | part_reset | gap_too_big).cumsum()
    return d


# ---- SS7: key-only grouping (no time bound) - reproduces the 22% collision
# rate from the markdown cell above ----
ss7_multi = ss7_raw[pd.to_numeric(ss7_raw["msg_parts"], errors="coerce").fillna(0) > 1].copy()
ss7_multi["time_stamp"] = pd.to_datetime(ss7_multi["time_stamp"])
print(f"SS7 multipart rows (msg_parts > 1): {len(ss7_multi)}")

key_cols = ["b_number", "msisdn", "sarref"]
key_only = ss7_multi.groupby(key_cols)
n_key_only = key_only.ngroups
n_colliding = key_only["sarref"].apply(lambda s: s.duplicated().any()).sum()
print(f"(b_number, msisdn, sarref) alone: {n_key_only} groups, "
      f"{n_colliding} still have a duplicate sarref (= a collision)")

# ---- SS7: same key + a 30s max-gap-between-parts bound ----
grouped = time_bounded_groups(
    ss7_multi, key_cols=key_cols, part_col="msg_part",
    time_col="time_stamp", max_gap=pd.Timedelta(seconds=30),
)
sizes = grouped.groupby("group_id").size()
n_colliding_bounded = grouped.groupby("group_id")["msg_part"].apply(lambda s: s.duplicated().any()).sum()
complete = grouped.groupby("group_id").apply(
    lambda g: sorted(g["msg_part"]) == list(range(1, int(g["msg_parts"].iloc[0]) + 1)),
    include_groups=False,
)
print(f"\n+ 30s max-gap bound: {sizes.shape[0]} groups, "
      f"{n_colliding_bounded} still colliding, "
      f"{complete.sum()} form a clean complete 1..N sequence "
      "(the rest are message_partial - a real incomplete delivery, a "
      "different concept from a grouping-key collision - see "
      "features/message_reassembly.py)")

# ---- SMPP: same check, sar_ref joined with oa/da instead ----
smpp_multi = smpp_raw[pd.to_numeric(smpp_raw["sar_msg_parts"], errors="coerce").fillna(0) > 1]
print(
    f"\nSMPP multipart rows (sar_msg_parts > 1): {len(smpp_multi)} "
    "-- NOTE: ingestion/smpp.py's clean() docstring says the SAR fields are "
    "almost always empty in real SMPP data (31/145k+ op-4 rows in the file "
    "checked there); most real SMPP multipart signal comes from the UDH "
    "concat IE inside `content` instead, not this field. A near-empty "
    "result below is expected, not a bug - ask if you want the UDH-based "
    "version of this same check."
)

if len(smpp_multi):
    smpp_multi = smpp_multi.copy()
    smpp_multi["time_stamp"] = pd.to_datetime(smpp_multi["time_stamp"])
    smpp_key = ["sar_ref", "oa", "da"]
    smpp_grouped = time_bounded_groups(
        smpp_multi, key_cols=smpp_key, part_col="sar_msg_part",
        time_col="time_stamp", max_gap=pd.Timedelta(seconds=30),
    )
    smpp_sizes = smpp_grouped.groupby("group_id").size()
    smpp_colliding = smpp_grouped.groupby("group_id")["sar_msg_part"].apply(lambda s: s.duplicated().any()).sum()
    print(f"(sar_ref, oa, da) + 30s max-gap bound: {smpp_sizes.shape[0]} groups, "
          f"{smpp_colliding} still colliding")

SS7 multipart rows (msg_parts > 1): 706910
(b_number, msisdn, sarref) alone: 230638 groups, 220410 still have a duplicate sarref (= a collision)


C:\Users\IshitaGodani\AppData\Local\Temp\ipykernel_31948\709832708.py:35: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  time_col="time_stamp", max_gap=pd.Timedelta(seconds=30),



+ 30s max-gap bound: 358487 groups, 0 still colliding, 208245 form a clean complete 1..N sequence (the rest are message_partial - a real incomplete delivery, a different concept from a grouping-key collision - see features/message_reassembly.py)

SMPP multipart rows (sar_msg_parts > 1): 380 -- NOTE: ingestion/smpp.py's clean() docstring says the SAR fields are almost always empty in real SMPP data (31/145k+ op-4 rows in the file checked there); most real SMPP multipart signal comes from the UDH concat IE inside `content` instead, not this field. A near-empty result below is expected, not a bug - ask if you want the UDH-based version of this same check.
(sar_ref, oa, da) + 30s max-gap bound: 380 groups, 0 still colliding


C:\Users\IshitaGodani\AppData\Local\Temp\ipykernel_31948\709832708.py:68: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  time_col="time_stamp", max_gap=pd.Timedelta(seconds=30),
